# Seasonality Feature Relevance Analysis

**Hypothesis**: Fitur seasonality (calendar + holiday) tidak memiliki korelasi signifikan dengan demand item tertentu (H0).

**Goals**:
- Hitung korelasi Spearman antara tiap fitur seasonality dan demand, per item.
- Uji signifikansi kolektif via permutation test.
- Ablation test: Twin-XGB tanpa vs dengan fitur seasonality.
- Segmentasi item berdasarkan strength seasonality.

In [ ]:
# ============================================================
# Rsync pull from local WSL via SSH tunnel (ngrok)
# ============================================================
# Update these values to your current ngrok host/port and user
NGROK_HOST = '0.tcp.ap.ngrok.io'
NGROK_PORT = 13231
REMOTE_USER = 'mwildanm'
REMOTE_PATH = '/home/mwildanm/fmcg-dynamic-pricing/data/raw/online_retail.csv'
LOCAL_DIR = '/content/data'
LOCAL_PATH = f'{LOCAL_DIR}/online_retail.csv'

!mkdir -p {LOCAL_DIR}
!rsync -avz -e "ssh -p {NGROK_PORT} -o StrictHostKeyChecking=no" {REMOTE_USER}@{NGROK_HOST}:{REMOTE_PATH} {LOCAL_PATH}


In [ ]:
# ============================================================
# Data path in Colab compute
# ============================================================
from pathlib import Path
RAW_PATH = '/content/data/online_retail.csv'
print(f'Using RAW_PATH: {RAW_PATH}')
if not Path(RAW_PATH).exists():
    raise FileNotFoundError(f'RAW_PATH not found: {RAW_PATH}')


In [ ]:
import gc
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
import holidays
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, f1_score, precision_score, recall_score
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)
pd.set_option('mode.copy_on_write', True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# --- Project root (optional) ---
# Not required for data loading in Colab compute
PROJECT_ROOT = Path.cwd()
print(f'PROJECT_ROOT: {PROJECT_ROOT}')


In [ ]:
# --- Project root (optional) ---
# Not required for data loading in Colab compute
from pathlib import Path
PROJECT_ROOT = Path.cwd()
print(f'PROJECT_ROOT: {PROJECT_ROOT}')


In [ ]:
!ls -la /content/data

In [ ]:
# ============================================================
# Config and memory-safe IO
# ============================================================
CHUNK_SIZE = 200_000
USECOLS = ['Invoice', 'StockCode', 'Quantity', 'InvoiceDate', 'Price', 'Country']
DTYPES = {
    'Invoice': 'string',
    'StockCode': 'string',
    'Quantity': 'float32',
    'Price': 'float32',
    'Country': 'string',
}

NON_PRODUCT_CODES = {
    'POST', 'DOT', 'C2', 'M', 'D', 'ADJUST', 'ADJUST2',
    'BANK CHARGES', 'AMAZONFEE', 'B', 'S', 'PADS',
    'TEST001', 'TEST002', 'GIFT_0001_10', 'GIFT_0001_20',
    'GIFT_0001_30', 'GIFT_0001_40', 'GIFT_0001_50', 'GIFT_0001_70',
    'GIFT_0001_80',
}


In [ ]:
def preprocess_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    df = chunk.rename(
        columns={
            'Invoice': 'invoice', 'StockCode': 'stock_code',
            'Quantity': 'quantity', 'InvoiceDate': 'invoice_date',
            'Price': 'price', 'Country': 'country',
        }
    ).copy()
    df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce')
    df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').astype('float32')
    df['price'] = pd.to_numeric(df['price'], errors='coerce').astype('float32')
    df['stock_code'] = df['stock_code'].astype('string').str.strip().str.upper()
    df['invoice'] = df['invoice'].astype('string').str.strip()
    df['country'] = df['country'].astype('string').str.strip()
    df = df.drop_duplicates()
    df = df[df['invoice_date'].notna()]
    df = df[df['stock_code'].notna() & (df['stock_code'] != '')]
    df = df[df['invoice'].notna() & (df['invoice'] != '')]
    df = df[~df['invoice'].str.startswith('C', na=False)]
    df = df[~df['stock_code'].isin(NON_PRODUCT_CODES)]
    df = df[(df['quantity'] > 0) & (df['price'] > 0)]
    df['date'] = df['invoice_date'].dt.normalize()
    df['revenue'] = df['quantity'] * df['price']
    df['stock_code'] = df['stock_code'].astype('category')
    df['country'] = df['country'].astype('category')
    df['invoice'] = df['invoice'].astype('category')
    return df[['stock_code', 'country', 'date', 'invoice', 'quantity', 'price', 'revenue']]

def aggregate_daily_from_chunks(path: Path) -> pd.DataFrame:
    daily_parts = []
    max_parts = 25
    for chunk in pd.read_csv(
        path, usecols=USECOLS, dtype=DTYPES, chunksize=CHUNK_SIZE, low_memory=False,
    ):
        cleaned = preprocess_chunk(chunk)
        daily = cleaned.groupby(['stock_code', 'country', 'date'], as_index=False, observed=True).agg(
            demand_qty=('quantity', 'sum'),
            revenue=('revenue', 'sum'),
            num_invoices=('invoice', 'nunique'),
            price_mean=('price', 'mean'),
        )
        daily_parts.append(daily)
        if len(daily_parts) >= max_parts:
            partial = pd.concat(daily_parts, ignore_index=True)
            daily_parts = [
                partial.groupby(['stock_code', 'country', 'date'], as_index=False, observed=True)
                .agg(demand_qty=('demand_qty', 'sum'), revenue=('revenue', 'sum'),
                     num_invoices=('num_invoices', 'sum'), price_mean=('price_mean', 'mean'))
            ]
            del partial; gc.collect()
        del chunk, cleaned, daily; gc.collect()
    daily_all = pd.concat(daily_parts, ignore_index=True)
    del daily_parts; gc.collect()
    daily_all = daily_all.groupby(['stock_code', 'country', 'date'], as_index=False, observed=True).agg(
        demand_qty=('demand_qty', 'sum'), revenue=('revenue', 'sum'),
        num_invoices=('num_invoices', 'sum'), price_mean=('price_mean', 'mean'),
    )
    daily_all['avg_price'] = np.where(
        daily_all['demand_qty'] > 0,
        daily_all['revenue'] / daily_all['demand_qty'], daily_all['price_mean'],
    )
    daily_all = daily_all.drop(columns=['price_mean'])
    for c in ['stock_code','country']:
        daily_all[c] = daily_all[c].astype('category')
    daily_all['demand_qty'] = daily_all['demand_qty'].astype('float32')
    daily_all['revenue'] = daily_all['revenue'].astype('float32')
    daily_all['avg_price'] = daily_all['avg_price'].astype('float32')
    daily_all['num_invoices'] = daily_all['num_invoices'].astype('float32')
    return daily_all

daily_df = aggregate_daily_from_chunks(RAW_PATH)
daily_df = daily_df.sort_values(['stock_code', 'country', 'date']).reset_index(drop=True)
print(f'daily_df: {daily_df.shape}')
daily_df.head()

In [ ]:
def build_full_daily_panel(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['stock_code', 'country', 'date']).reset_index(drop=True)
    def _resample(group: pd.DataFrame) -> pd.DataFrame:
        group = group.set_index('date').asfreq('D')
        group['stock_code'] = group['stock_code'].iloc[0]
        group['country'] = group['country'].iloc[0]
        return group.reset_index()
    panel = df.groupby(['stock_code', 'country'], group_keys=False, sort=False).apply(_resample)
    panel = panel.reset_index(drop=True)
    fill_cols = ['demand_qty', 'revenue', 'num_invoices']
    panel[fill_cols] = panel[fill_cols].fillna(0)
    panel['avg_price'] = panel['avg_price'].astype('float32')
    panel['avg_price'] = panel.groupby(['stock_code', 'country'], sort=False)['avg_price'].ffill().bfill().fillna(0)
    for c in ['stock_code','country']:
        panel[c] = panel[c].astype('category')
    panel['demand_qty'] = panel['demand_qty'].astype('float32')
    panel['revenue'] = panel['revenue'].astype('float32')
    panel['avg_price'] = panel['avg_price'].astype('float32')
    panel['num_invoices'] = panel['num_invoices'].astype('float32')
    return panel

panel_df = build_full_daily_panel(daily_df)
del daily_df; gc.collect()
panel_df = panel_df.sort_values(['stock_code', 'country', 'date']).reset_index(drop=True)
print(f'panel_df: {panel_df.shape}')
panel_df.head()

In [ ]:
COUNTRY_TO_HOLIDAYS = {
    'Australia': 'AU', 'Austria': 'AT', 'Bahrain': 'BH', 'Belgium': 'BE',
    'Bermuda': 'BM', 'Brazil': 'BR', 'Canada': 'CA', 'Channel Islands': 'GB',
    'Cyprus': 'CY', 'Czech Republic': 'CZ', 'Denmark': 'DK', 'EIRE': 'IE',
    'European Community': None, 'Finland': 'FI', 'France': 'FR', 'Germany': 'DE',
    'Greece': 'GR', 'Hong Kong': 'HK', 'Iceland': 'IS', 'Israel': 'IL',
    'Italy': 'IT', 'Japan': 'JP', 'Korea': 'KR', 'Lebanon': 'LB',
    'Lithuania': 'LT', 'Malta': 'MT', 'Netherlands': 'NL', 'Nigeria': 'NG',
    'Norway': 'NO', 'Poland': 'PL', 'Portugal': 'PT', 'RSA': 'ZA',
    'Saudi Arabia': 'SA', 'Singapore': 'SG', 'Spain': 'ES', 'Sweden': 'SE',
    'Switzerland': 'CH', 'Thailand': 'TH', 'USA': 'US', 'United Arab Emirates': 'AE',
    'United Kingdom': 'GB', 'Unspecified': None, 'West Indies': None,
}

panel_df['country_code'] = panel_df['country'].map(COUNTRY_TO_HOLIDAYS).astype('category')

years = panel_df['date'].dt.year.unique().tolist()
holiday_rows = []
supported = set(holidays.list_supported_countries())
for code in sorted(panel_df['country_code'].dropna().astype(str).unique()):
    if code not in supported: continue
    holiday_set = holidays.country_holidays(code, years=years)
    holiday_rows.append(pd.DataFrame({
        'country_code': code, 'date': list(holiday_set.keys()), 'is_hari_besar': 1,
    }))
holiday_df = pd.concat(holiday_rows, ignore_index=True) if holiday_rows else pd.DataFrame(columns=['country_code', 'date', 'is_hari_besar'])
holiday_df['date'] = pd.to_datetime(holiday_df['date'], errors='coerce')

panel_df = panel_df.merge(holiday_df, on=['country_code', 'date'], how='left', copy=False)
if 'is_hari_besar' not in panel_df.columns:
    panel_df['is_hari_besar'] = 0
panel_df['is_hari_besar'] = panel_df['is_hari_besar'].fillna(0).astype('uint8')

pre_holiday_rows = []
if not holiday_df.empty:
    for offset in [1, 2, 3]:
        pre_holiday_rows.append(holiday_df.assign(
            date=holiday_df['date'] - pd.Timedelta(days=offset), is_pre_hari_besar=1,
        )[['country_code', 'date', 'is_pre_hari_besar']])
pre_holiday_df = pd.concat(pre_holiday_rows, ignore_index=True) if pre_holiday_rows else pd.DataFrame(columns=['country_code', 'date', 'is_pre_hari_besar'])
pre_holiday_df = pre_holiday_df.drop_duplicates()
panel_df = panel_df.merge(pre_holiday_df, on=['country_code', 'date'], how='left', copy=False)
del pre_holiday_df, pre_holiday_rows, holiday_df, holiday_rows; gc.collect()
panel_df['is_pre_hari_besar'] = panel_df['is_pre_hari_besar'].fillna(0).astype('uint8')

panel_df['day_of_week'] = panel_df['date'].dt.dayofweek.astype('int8')
panel_df['week_of_year'] = panel_df['date'].dt.isocalendar().week.astype('int16')
panel_df['month'] = panel_df['date'].dt.month.astype('int8')
panel_df['quarter'] = panel_df['date'].dt.quarter.astype('int8')
panel_df['day_of_month'] = panel_df['date'].dt.day.astype('int8')
panel_df['is_weekend'] = (panel_df['day_of_week'] >= 5).astype('uint8')
panel_df['is_month_start'] = panel_df['date'].dt.is_month_start.astype('uint8')
panel_df['is_month_end'] = panel_df['date'].dt.is_month_end.astype('uint8')
month_end = panel_df['date'] + pd.offsets.MonthEnd(0)
panel_df['days_to_month_end'] = (month_end - panel_df['date']).dt.days.astype('int16')
panel_df['week_of_month'] = ((panel_df['date'].dt.day - 1) // 7 + 1).astype('int8')
panel_df['is_month_start_window'] = (panel_df['date'].dt.day <= 5).astype('uint8')
panel_df['is_month_end_window'] = (panel_df['date'].dt.day >= 25).astype('uint8')
gc.collect()
print(f'After holiday/calendar: {panel_df.shape}')
panel_df.head()

In [ ]:
group_cols = ['stock_code', 'country']
panel_df = panel_df.sort_values(group_cols + ['date']).reset_index(drop=True)

demand_shifted = panel_df.groupby(group_cols)['demand_qty'].shift(1)
panel_df['demand_lag_1'] = demand_shifted
panel_df['demand_lag_2'] = panel_df.groupby(group_cols)['demand_qty'].shift(2)
panel_df['demand_lag_7'] = panel_df.groupby(group_cols)['demand_qty'].shift(7)
panel_df['demand_lag_14'] = panel_df.groupby(group_cols)['demand_qty'].shift(14)
panel_df['demand_lag_21'] = panel_df.groupby(group_cols)['demand_qty'].shift(21)
panel_df['demand_lag_28'] = panel_df.groupby(group_cols)['demand_qty'].shift(28)
panel_df['demand_lag_35'] = panel_df.groupby(group_cols)['demand_qty'].shift(35)
panel_df['demand_lag_56'] = panel_df.groupby(group_cols)['demand_qty'].shift(56)
panel_df['demand_lag_84'] = panel_df.groupby(group_cols)['demand_qty'].shift(84)

panel_df['roll_max_7'] = demand_shifted.rolling(window=7, min_periods=1).max().values
panel_df['roll_max_28'] = demand_shifted.rolling(window=28, min_periods=1).max().values
panel_df['roll_zero_count_14'] = (demand_shifted == 0).rolling(window=14, min_periods=1).sum().values
panel_df['roll_mean_7'] = demand_shifted.rolling(window=7, min_periods=1).mean().values
panel_df['roll_mean_14'] = demand_shifted.rolling(window=14, min_periods=1).mean().values
panel_df['roll_mean_28'] = demand_shifted.rolling(window=28, min_periods=1).mean().values
panel_df['roll_mean_56'] = demand_shifted.rolling(window=56, min_periods=1).mean().values
panel_df['roll_median_7'] = demand_shifted.rolling(window=7, min_periods=1).median().values
panel_df['roll_median_14'] = demand_shifted.rolling(window=14, min_periods=1).median().values
panel_df['roll_median_28'] = demand_shifted.rolling(window=28, min_periods=1).median().values
panel_df['roll_std_7'] = demand_shifted.rolling(window=7, min_periods=1).std().values
panel_df['roll_std_14'] = demand_shifted.rolling(window=14, min_periods=1).std().values
panel_df['roll_std_28'] = demand_shifted.rolling(window=28, min_periods=1).std().values
panel_df['roll_std_56'] = demand_shifted.rolling(window=56, min_periods=1).std().values
panel_df['roll_max_56'] = demand_shifted.rolling(window=56, min_periods=1).max().values
panel_df['roll_max_84'] = demand_shifted.rolling(window=84, min_periods=1).max().values

roll_mean_3 = demand_shifted.rolling(window=3, min_periods=1).mean().values
roll_mean_14 = demand_shifted.rolling(window=14, min_periods=1).mean().values
panel_df['demand_acceleration_3d'] = roll_mean_3 / (roll_mean_14 + 1e-8)
panel_df['spike_ratio_28'] = panel_df['roll_max_28'] / (panel_df['roll_mean_28'] + 1e-8)
panel_df['spike_ratio_56'] = panel_df['roll_max_56'] / (panel_df['roll_mean_56'] + 1e-8)

panel_df['pct_change_1'] = (panel_df['demand_lag_1'] - panel_df['demand_lag_2']) / (panel_df['demand_lag_2'] + 1e-8)
panel_df['pct_change_7'] = (panel_df['demand_lag_7'] - panel_df['demand_lag_14']) / (panel_df['demand_lag_14'] + 1e-8)

last_sale_date = panel_df['date'].where(demand_shifted > 0)
last_sale_date = last_sale_date.groupby(panel_df[group_cols].apply(tuple, axis=1)).ffill()
panel_df['days_since_last_sale'] = (panel_df['date'] - last_sale_date).dt.days
panel_df['days_since_last_sale'] = panel_df['days_since_last_sale'].fillna(9999).astype('int16')

price_shifted = panel_df.groupby(group_cols)['avg_price'].shift(1)
roll_price_max_30 = price_shifted.rolling(window=30, min_periods=1).max().values
panel_df['discount_depth_pct'] = (roll_price_max_30 - panel_df['avg_price']) / (roll_price_max_30 + 1e-8)
roll_price_mean_14 = price_shifted.rolling(window=14, min_periods=1).mean().values
panel_df['price_momentum'] = panel_df['avg_price'] / (roll_price_mean_14 + 1e-8)

numeric_cols = panel_df.select_dtypes(include=['number']).columns
panel_df[numeric_cols] = panel_df[numeric_cols].replace([np.inf, -np.inf], 0).fillna(0)
for col in panel_df.select_dtypes(include=['float64']).columns:
    panel_df[col] = panel_df[col].astype('float32')
gc.collect()
print(f'After lag/rolling: {panel_df.shape}')

# Cleanup intermediates
del demand_shifted, price_shifted, roll_mean_3,\
    roll_mean_14, last_sale_date, roll_price_max_30,\
    roll_price_mean_14
gc.collect()


In [ ]:
SEASONALITY_FEATURES = [
    'day_of_week', 'week_of_year', 'month', 'quarter', 'day_of_month',
    'is_weekend', 'is_month_start', 'is_month_end',
    'days_to_month_end', 'week_of_month',
    'is_month_start_window', 'is_month_end_window',
    'is_hari_besar', 'is_pre_hari_besar',
]

NON_SEASONALITY_FEATURES = [
    'demand_lag_1', 'demand_lag_2', 'demand_lag_7', 'demand_lag_14',
    'demand_lag_21', 'demand_lag_28', 'demand_lag_35', 'demand_lag_56', 'demand_lag_84',
    'days_since_last_sale', 'roll_zero_count_14',
    'roll_max_7', 'roll_max_28',
    'roll_mean_7', 'roll_mean_14', 'roll_mean_28', 'roll_mean_56',
    'roll_median_7', 'roll_median_14', 'roll_median_28',
    'roll_std_7', 'roll_std_14', 'roll_std_28', 'roll_std_56',
    'roll_max_56', 'roll_max_84',
    'demand_acceleration_3d',
    'spike_ratio_28', 'spike_ratio_56',
    'pct_change_1', 'pct_change_7',
    'discount_depth_pct', 'price_momentum',
]

ALL_FEATURES = SEASONALITY_FEATURES + NON_SEASONALITY_FEATURES

TARGET_COL = 'demand_qty'
PRICE_COL = 'avg_price'
DATE_COL = 'date'

print(f'Seasonality features ({len(SEASONALITY_FEATURES)}): {SEASONALITY_FEATURES}')
print(f'Non-seasonality features ({len(NON_SEASONALITY_FEATURES)}): {len(NON_SEASONALITY_FEATURES)}')
print(f'Total features: {len(ALL_FEATURES)}')

In [ ]:
MIN_OBS = 60
item_obs = panel_df.groupby(['stock_code', 'country']).size()
valid_items = item_obs[item_obs >= MIN_OBS].index
panel_df = panel_df[panel_df.set_index(['stock_code', 'country']).index.isin(valid_items)].copy()
panel_df = panel_df.sort_values(group_cols + ['date']).reset_index(drop=True)
print(f'After filter (min_obs={MIN_OBS}): {panel_df.shape}')
print(f'Unique items: {panel_df[["stock_code","country"]].drop_duplicates().shape[0]}')

In [ ]:
def time_series_folds(dates, horizon_days=30, n_splits=3, min_train_days=180):
    dates = np.array(sorted(pd.to_datetime(dates).unique()))
    total_days = len(dates)
    splits = []
    for i in range(n_splits):
        val_end_idx = total_days - (n_splits - i - 1) * horizon_days
        val_start_idx = val_end_idx - horizon_days
        train_end_idx = val_start_idx - 1
        if train_end_idx < min_train_days: continue
        train_end = dates[train_end_idx]
        val_start = dates[val_start_idx]
        val_end = dates[val_end_idx - 1]
        splits.append((train_end, val_start, val_end))
    return splits

splits = time_series_folds(panel_df[DATE_COL], horizon_days=30, n_splits=3, min_train_days=180)
splits

In [ ]:
def calc_cls(y_true, y_pred, avg_price_arr, margin=0.20):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    avg_price = np.asarray(avg_price_arr, dtype=float)
    return float(np.sum(np.maximum(y_true - y_pred, 0) * avg_price * margin))

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    if mask.sum() == 0: return 0.0
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100)

def evaluate_prediction(y_true, y_pred, avg_price_arr, baseline_mae=None):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))
    smape_val = smape(y_true, y_pred)
    y_true_bin = (np.asarray(y_true) > 0).astype(int)
    y_pred_bin = (np.asarray(y_pred) > 0).astype(int)
    f1_zero = f1_score(y_true_bin, y_pred_bin)
    precision_zero = precision_score(y_true_bin, y_pred_bin, zero_division=0)
    recall_zero = recall_score(y_true_bin, y_pred_bin, zero_division=0)
    cls = calc_cls(y_true, y_pred, avg_price_arr)
    ofr = np.minimum(y_true, y_pred).sum() / (y_true.sum() + 1e-8)
    oos_rate = float(np.mean(y_pred < y_true))
    fva = None
    if baseline_mae is not None and baseline_mae > 0:
        fva = float((baseline_mae - mae) / baseline_mae)
    return {'mae': mae, 'rmse': rmse, 'smape': smape_val, 'f1_zero': f1_zero,
            'precision_zero': precision_zero, 'recall_zero': recall_zero,
            'cls': cls, 'ofr': ofr, 'oos_rate': oos_rate, 'fva': fva}

def asymmetric_obj(alpha=2.0):
    def obj(y_true, y_pred, sample_weight=None):
        residual = y_pred - y_true
        grad = np.where(residual > 0, 2 * residual, 2 * alpha * residual)
        hess = np.where(residual > 0, 2.0, 2.0 * alpha)
        if sample_weight is not None:
            grad = grad * sample_weight
            hess = hess * sample_weight
        return grad, hess
    return obj

def quantile_obj(q=0.8):
    def obj(y_true, y_pred, sample_weight=None):
        residual = y_pred - y_true
        grad = np.where(residual >= 0, q, q - 1.0)
        hess = np.ones_like(grad) * 1e-6
        if sample_weight is not None:
            grad = grad * sample_weight
            hess = hess * sample_weight
        return grad, hess
    return obj

## A. Korelasi Spearman per Item (Seasonality vs Demand)

Untuk setiap item, hitung korelasi Spearman antara tiap fitur seasonality dan `demand_qty`.
Jika mayoritas item punya |rho| < 0.2, ini mendukung H0 (tidak ada korelasi).

In [ ]:
RHO_THRESHOLD = 0.2

item_corr_rows = []
items = panel_df[group_cols].drop_duplicates().to_numpy()
print(f'Computing Spearman correlation for {len(items)} items...')

for idx, (sk, co) in enumerate(items):
    mask = (panel_df['stock_code'] == sk) & (panel_df['country'] == co)
    sub = panel_df.loc[mask, SEASONALITY_FEATURES + [TARGET_COL]]
    if len(sub) < MIN_OBS:
        continue
    for feat in SEASONALITY_FEATURES:
        rho_all, p_all = spearmanr(sub[feat], sub[TARGET_COL])
        nz = sub[sub[TARGET_COL] > 0]
        if len(nz) >= 10:
            rho_nz, p_nz = spearmanr(nz[feat], nz[TARGET_COL])
        else:
            rho_nz, p_nz = np.nan, np.nan
        item_corr_rows.append({
            'stock_code': sk, 'country': co,
            'feature': feat,
            'rho_all': rho_all, 'p_all': p_all,
            'rho_nonzero': rho_nz, 'p_nonzero': p_nz,
            'n_days': len(sub), 'n_nonzero': len(nz),
        })
    if (idx + 1) % 500 == 0:
        print(f'  processed {idx + 1}/{len(items)} items')

corr_df = pd.DataFrame(item_corr_rows)
print(f'\nTotal correlation rows: {len(corr_df)}')
corr_df.head()

In [ ]:
# Aggregate by feature
feat_summary = corr_df.groupby('feature').agg(
    mean_rho_all=('rho_all', 'mean'),
    median_rho_all=('rho_all', 'median'),
    std_rho_all=('rho_all', 'std'),
    pct_ge_threshold=('rho_all', lambda x: (np.abs(x) >= RHO_THRESHOLD).mean()),
    pct_significant=('p_all', lambda x: (x < 0.05).mean()),
    mean_rho_nonzero=('rho_nonzero', 'mean'),
    median_rho_nonzero=('rho_nonzero', 'median'),
).reset_index().round(4)
feat_summary.sort_values('pct_ge_threshold', ascending=False)

In [ ]:
total_pairs = len(corr_df)
sig_pairs = (corr_df['rho_all'].abs() >= RHO_THRESHOLD).sum()
print(f'Total item x feature pairs: {total_pairs}')
print(f'Pairs with |rho| >= {RHO_THRESHOLD}: {sig_pairs} ({sig_pairs/total_pairs*100:.1f}%)')
print(f'Pairs with p < 0.05: {(corr_df["p_all"] < 0.05).sum()} ({(corr_df["p_all"] < 0.05).mean()*100:.1f}%)')
print()
sig_items = corr_df.groupby(['stock_code','country']).filter(
    lambda g: (g['rho_all'].abs() >= RHO_THRESHOLD).any()
)[['stock_code','country']].drop_duplicates().shape[0]
total_items = corr_df[['stock_code','country']].drop_duplicates().shape[0]
print(f'Items with at least one strong correlation: {sig_items}/{total_items} ({sig_items/total_items*100:.1f}%)')

## B. Permutation Test

Shuffle demand per item, hitung distribusi rho permutasi.
Bandingkan mean |rho| aktual vs permutasi untuk menguji signifikansi kolektif.

In [ ]:
# ==========================================================
# Optimized Permutation Test
# Strategi:
#   1) Precompute per-item arrays (X seasonality + y) sekali saja
#   2) Precompute rank(X) sekali (Spearman via Pearson on ranks)
#   3) Permutasi hanya shuffle y + rank ulang y
#   4) Vectorized correlation (dot product) untuk semua fitur sekaligus
#   5) Parallel per permutation (thread, tanpa pickle overhead)
# ==========================================================
import time
from joblib import Parallel, delayed

N_PERMS = 100
N_JOBS = -1  # gunakan semua core
SEED = 42

# --------------------------------------------------
# Precompute per-item arrays + rank(X)
# --------------------------------------------------
print('Precomputing per-item arrays...', flush=True)
items_arr = panel_df[group_cols].drop_duplicates().to_numpy()
item_slices = []  # list of (X, y, rank_X) for faster iteration
total_items = len(items_arr)
for idx, (sk, co) in enumerate(items_arr):
    mask = (panel_df['stock_code'] == sk) & (panel_df['country'] == co)
    sub = panel_df.loc[mask, SEASONALITY_FEATURES + [TARGET_COL]]
    if len(sub) < MIN_OBS:
        continue
    X = sub[SEASONALITY_FEATURES].to_numpy(dtype=np.float32, copy=False)
    y = sub[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
    # Precompute rank(X) — statis antar permutasi
    rank_X = np.argsort(X, axis=0).argsort(axis=0).astype(np.float32)
    item_slices.append((X, y, rank_X))
    if (idx + 1) % 1000 == 0:
        print(f'  [{idx+1}/{total_items}] precomputed', flush=True)

print(f'Cached {len(item_slices)} items for permutation.')
del items_arr

# --------------------------------------------------
# Precompute observed mean |rho|
# --------------------------------------------------
print('Computing observed (actual) mean |rho|...', flush=True)
obs_rho_sums = {feat: 0.0 for feat in SEASONALITY_FEATURES}
n_items = len(item_slices)
for X, y, rank_X in item_slices:
    rank_y = np.argsort(y).argsort().astype(np.float32)
    ry_centered = rank_y - np.mean(rank_y)
    ry_std = ry_centered.std()
    n = float(len(y))
    for j, feat in enumerate(SEASONALITY_FEATURES):
        rx_centered = rank_X[:, j] - np.mean(rank_X[:, j])
        denom = n * rx_centered.std() * ry_std
        rho = np.dot(rx_centered, ry_centered) / denom if denom > 0 else 0.0
        obs_rho_sums[feat] += abs(rho)

actual_mean_abs_rho = {feat: v/n_items for feat, v in obs_rho_sums.items()}
print(f'Observed rhos computed for {len(SEASONALITY_FEATURES)} features.')
del obs_rho_sums

# --------------------------------------------------
# Permutation function (satu permutasi)
# --------------------------------------------------
def _one_perm(perm_i, seed_offset):
    local_rng = np.random.default_rng(SEED + seed_offset)
    sum_abs_rho = np.zeros(len(SEASONALITY_FEATURES), dtype=np.float64)
    for X, y, rank_X in item_slices:
        y_shuffled = local_rng.permutation(y)
        rank_y = np.argsort(y_shuffled).argsort().astype(np.float32)
        ry_centered = rank_y - rank_y.mean()
        ry_std = ry_centered.std()
        n = float(len(y))
        for j in range(len(SEASONALITY_FEATURES)):
            rx_centered = rank_X[:, j] - rank_X[:, j].mean()
            denom = n * rx_centered.std() * ry_std
            if denom > 0:
                sum_abs_rho[j] += np.dot(rx_centered, ry_centered) / denom
    return sum_abs_rho / len(item_slices)

# --------------------------------------------------
# Run parallel permutations
# --------------------------------------------------
print(f'Running {N_PERMS} permutations (parallel, {N_JOBS} workers)...')
t0 = time.time()
results = Parallel(n_jobs=N_JOBS, prefer='threads')(
    delayed(_one_perm)(i, i) for i in range(N_PERMS)
)
elapsed = time.time() - t0
print(f'Done in {elapsed:.1f}s ({elapsed/N_PERMS:.1f}s per perm).\n')

# --------------------------------------------------
# Aggregate results
# --------------------------------------------------
perm_means_by_feat = {feat: [] for feat in SEASONALITY_FEATURES}
for perm_rhos in results:
    for j, feat in enumerate(SEASONALITY_FEATURES):
        perm_means_by_feat[feat].append(perm_rhos[j])

perm_summary_rows = []
for j, feat in enumerate(SEASONALITY_FEATURES):
    actual = actual_mean_abs_rho[feat]
    perm_means = perm_means_by_feat[feat]
    p_value = float(np.mean(np.array(perm_means) >= actual))
    perm_summary_rows.append({
        'feature': feat,
        'actual_mean_abs_rho': round(actual, 4),
        'perm_mean_abs_rho': round(float(np.mean(perm_means)), 4),
        'perm_ci_lower': round(float(np.percentile(perm_means, 2.5)), 4),
        'perm_ci_upper': round(float(np.percentile(perm_means, 97.5)), 4),
        'p_value': round(p_value, 4),
    })

perm_df = pd.DataFrame(perm_summary_rows)

# Cleanup large intermediates
del item_slices, results, perm_means_by_feat

perm_df


In [ ]:
nonsig = perm_df[perm_df['p_value'] > 0.05].shape[0]
sig = perm_df[perm_df['p_value'] <= 0.05].shape[0]
print(f'Features with p <= 0.05: {sig}/{len(perm_df)} (reject H0)')
print(f'Features with p > 0.05: {nonsig}/{len(perm_df)} (fail to reject H0)')
print()
for _, row in perm_df.iterrows():
    verdict = 'SIGNIFICANT' if row['p_value'] <= 0.05 else 'not significant'
    print(f"  {row['feature']:25s} actual={row['actual_mean_abs_rho']:.4f} perm={row['perm_mean_abs_rho']:.4f} p={row['p_value']:.4f} -> {verdict}")

## C. Twin-XGB Ablation Test

Bandingkan model **tanpa** vs **dengan** fitur seasonality.
Fokus pada metrik: MAE, RMSE, CLS, OFR, terutama di bucket 95-100%.

In [ ]:
CLF_PARAMS = {
    'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.05,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'tree_method': 'hist', 'max_bin': 256,
    'random_state': RANDOM_STATE, 'n_jobs': -1,
}
REG_PARAMS = {
    'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.05,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'tree_method': 'hist', 'max_bin': 256,
    'random_state': RANDOM_STATE, 'n_jobs': -1,
}

ALPHA_UNDER = 50.0
QUANTILE_Q = 0.8
QUANTILE_Q_TOP = 0.9
THRESHOLD_GRID = np.round(np.arange(0.10, 0.91, 0.05), 2).tolist()
USE_LOG_TARGET = True
TOP_SEGMENT_PCT = 0.05
SAMPLE_WEIGHT_ALPHA = 4.0
SAMPLE_WEIGHT_CAP = 5.0

In [ ]:
def run_twin_xgb(feature_cols, label, panel_df, splits):
    """Run standard twin-xgb CV returning aggregated metrics."""
    fold_rows = []
    for fold_idx, (train_end, val_start, val_end) in enumerate(splits, start=1):
        train_mask = panel_df[DATE_COL] <= train_end
        val_mask = (panel_df[DATE_COL] >= val_start) & (panel_df[DATE_COL] <= val_end)

        train_segment = (
            panel_df.loc[train_mask]
            .groupby(['stock_code', 'country'])[TARGET_COL]
            .sum().sort_values(ascending=False)
        )
        top_n = max(1, int(len(train_segment) * TOP_SEGMENT_PCT))
        top_keys = set(train_segment.head(top_n).index)

        df_train = panel_df.loc[train_mask, feature_cols + [TARGET_COL, PRICE_COL, 'stock_code', 'country']]
        df_val = panel_df.loc[val_mask, feature_cols + [TARGET_COL, PRICE_COL, 'stock_code', 'country']]

        X_train = df_train[feature_cols].to_numpy(dtype=np.float32, copy=False)
        y_train = df_train[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
        X_val = df_val[feature_cols].to_numpy(dtype=np.float32, copy=False)
        y_val = df_val[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
        price_val = df_val[PRICE_COL].to_numpy(dtype=np.float32, copy=False)
        train_keys = list(zip(df_train['stock_code'].to_numpy(), df_train['country'].to_numpy()))
        val_keys = list(zip(df_val['stock_code'].to_numpy(), df_val['country'].to_numpy()))
        val_is_top = np.array([key in top_keys for key in val_keys])

        y_train_zero = (y_train > 0).astype(int)
        scale_pos_weight = (len(y_train_zero) - y_train_zero.sum()) / (y_train_zero.sum() + 1e-8)
        clf = xgb.XGBClassifier(**CLF_PARAMS, objective='binary:logistic', scale_pos_weight=scale_pos_weight)
        clf.fit(X_train, y_train_zero)
        calibrator = CalibratedClassifierCV(clf, method='isotonic', cv=3)
        calibrator.fit(X_train, y_train_zero)

        nonzero_mask = y_train > 0
        y_train_nonzero = y_train[nonzero_mask]
        y_train_reg = np.log1p(y_train_nonzero) if USE_LOG_TARGET else y_train_nonzero
        q95 = np.quantile(y_train_nonzero, 0.95) if y_train_nonzero.size else 0.0
        weight_scale = (y_train_nonzero / (q95 + 1e-8)) if q95 > 0 else np.zeros_like(y_train_nonzero)
        sample_weight = 1.0 + SAMPLE_WEIGHT_ALPHA * np.minimum(weight_scale, SAMPLE_WEIGHT_CAP)

        reg_top = xgb.XGBRegressor(**REG_PARAMS, objective='reg:quantileerror', quantile_alpha=QUANTILE_Q_TOP)
        reg_tail = xgb.XGBRegressor(**REG_PARAMS, objective=asymmetric_obj(ALPHA_UNDER))
        reg_top.fit(X_train[nonzero_mask], y_train_reg, sample_weight=sample_weight)
        reg_tail.fit(X_train[nonzero_mask], y_train_reg)

        prob_nonzero = calibrator.predict_proba(X_val)[:, 1]
        pred_reg_top = reg_top.predict(X_val)
        pred_reg_tail = reg_tail.predict(X_val)
        if USE_LOG_TARGET:
            pred_reg_top = np.expm1(pred_reg_top)
            pred_reg_tail = np.expm1(pred_reg_tail)
        pred_reg_top = np.maximum(pred_reg_top, 0)
        pred_reg_tail = np.maximum(pred_reg_tail, 0)
        pred_reg = np.where(val_is_top, pred_reg_top, pred_reg_tail)

        # Bias correction
        if val_is_top.any():
            residual_top = pred_reg_top[val_is_top] - y_val[val_is_top]
            q95 = np.quantile(y_val[val_is_top], 0.95)
            tail_mask = y_val[val_is_top] >= q95
            tail_residual = residual_top[tail_mask] if tail_mask.any() else residual_top
            tail_mean = np.mean(y_val[val_is_top][tail_mask]) if tail_mask.any() else np.mean(y_val[val_is_top])
            top_beta = max(0.0, -np.mean(tail_residual) / (tail_mean + 1e-8))
            pred_reg = np.where(val_is_top, pred_reg * (1.0 + top_beta), pred_reg)

        baseline_mae = float(np.mean(np.abs(y_val)))
        best_threshold, best_cls, best_metrics = None, None, None
        for threshold in THRESHOLD_GRID:
            pred_zero = (prob_nonzero >= threshold).astype(int)
            pred = pred_reg * pred_zero
            metrics = evaluate_prediction(y_val, pred, price_val, baseline_mae=baseline_mae)
            if best_cls is None or metrics['cls'] < best_cls:
                best_cls = metrics['cls']
                best_threshold = threshold
                best_metrics = metrics

        best_metrics['fold'] = fold_idx
        best_metrics['threshold'] = best_threshold
        fold_rows.append(best_metrics)

        del df_train, df_val, X_train, y_train, X_val, y_val, price_val,\
            clf, reg_top, reg_tail, calibrator, pred_reg
        gc.collect()

    df = pd.DataFrame(fold_rows)
    agg = df.mean(numeric_only=True).to_dict()
    agg['label'] = label
    return agg, df

In [ ]:
print('Running baseline (non-seasonality only)...')
agg_baseline, df_baseline = run_twin_xgb(NON_SEASONALITY_FEATURES, 'baseline', panel_df, splits)
print('Running full (all features)...')
agg_full, df_full = run_twin_xgb(ALL_FEATURES, 'full', panel_df, splits)
print()
ablation_results = pd.DataFrame([agg_baseline, agg_full]).set_index('label').round(4)
ablation_results[['mae','rmse','smape','cls','ofr','oos_rate']]

In [ ]:
# Delta: full - baseline
delta = {}
for col in ['mae','rmse','smape','cls','ofr','oos_rate']:
    delta[col] = agg_full[col] - agg_baseline[col]
    delta_pct = delta[col] / abs(agg_baseline[col]) * 100 if agg_baseline[col] != 0 else 0
    delta[f'{col}_pct'] = delta_pct
delta_df = pd.DataFrame([delta]).round(4).T
delta_df.columns = ['delta']
delta_df['sign'] = delta_df['delta'].apply(
    lambda x: 'IMPROVEMENT' if x < 0 else 'DEGRADATION' if x > 0 else 'SAME')
delta_df

In [ ]:
def get_quantile_metrics(df, label):
    rows = []
    sub = df.copy()
    sub = sub[sub['actual'] > 0].copy()
    if sub.empty:
        return pd.DataFrame()
    sub['q_bucket'] = pd.qcut(
        sub['actual'],
        q=[0.0, 0.5, 0.8, 0.95, 1.0],
        labels=['0-50','50-80','80-95','95-100'],
        duplicates='drop',
    )
    for bucket, grp in sub.groupby('q_bucket'):
        m = evaluate_prediction(grp['actual'], grp['forecast'], grp[PRICE_COL])
        m['bucket'] = bucket
        rows.append(m)
    qdf = pd.DataFrame(rows)
    qdf['label'] = label
    return qdf


## D. Segmentasi Berdasarkan Strength Seasonality

Kelompokkan item menjadi strong vs weak seasonality. Bandingkan error.

In [ ]:
item_strength = corr_df.groupby(['stock_code','country']).agg(
    strong_features=('rho_all', lambda x: (np.abs(x) >= RHO_THRESHOLD).sum()),
    total_features=('rho_all', 'count'),
).reset_index()
item_strength['strength_ratio'] = item_strength['strong_features'] / item_strength['total_features']
item_strength['segment'] = np.where(item_strength['strength_ratio'] >= 0.25, 'strong', 'weak')
print(item_strength['segment'].value_counts())
item_strength.head()

In [ ]:
strong_set = set(item_strength.loc[item_strength['segment']=='strong', ['stock_code','country']].itertuples(index=False, name=None))
weak_set = set(item_strength.loc[item_strength['segment']=='weak', ['stock_code','country']].itertuples(index=False, name=None))

pred_full['key'] = list(zip(pred_full['stock_code'], pred_full['country']))
seg_rows = []
for seg_name, key_set in [('strong', strong_set), ('weak', weak_set)]:
    seg_pred = pred_full[pred_full['key'].isin(key_set)]
    if seg_pred.empty:
        continue
    m = evaluate_prediction(seg_pred['actual'], seg_pred['forecast'], seg_pred[PRICE_COL])
    m['segment'] = seg_name
    m['n_items'] = seg_pred[['stock_code','country']].drop_duplicates().shape[0]
    seg_rows.append(m)
seg_df = pd.DataFrame(seg_rows).round(4)
seg_df[['segment','n_items','mae','rmse','cls','ofr']]


## E. Summary & Interpretasi

**Correlation Analysis**:
- Proporsi item x feature pairs dengan |rho| >= 0.2.
- Proporsi item dengan minimal satu korelasi signifikan.

**Permutation Test**:
- Fitur seasonality mana yang secara kolektif signifikan.

**Ablation Test**:
- Delta MAE/RMSE/CLS/OFR baseline to full.
- Apakah seasonality features memberikan kontribusi nyata?

**Segmentation**:
- Apakah item strong-seasonality punya error berbeda dari weak-seasonality?

Kesimpulan akhir akan menentukan apakah fitur seasonality layak dipertahankan,
diseleksi subset-nya, atau di-drop sebagian.